In [ ]:
# 1. Verify the upload: size and checksum must match the local file
!ls -lh /content/house_yolo.tar
!md5sum /content/house_yolo.tar


-rw-r--r-- 1 root root 521M Nov 16 21:44 /content/house_yolo.tar
9e056b5a4d1c56749eca9a7cec0150b1  /content/house_yolo.tar


In [ ]:
# 2. Install Ultralytics, unpack the dataset, point data.yaml at the new location
!pip install -q ultralytics
!tar -xf /content/house_yolo.tar -C /content/
!sed -i 's#^path:.*#path: /content/house_yolo#' /content/house_yolo/data.yaml
!cat /content/house_yolo/data.yaml
!ls /content/house_yolo/images/train | wc -l     # expect 1998


path: /content/house_yolo
train: images/train
val: images/val
test: images/test
names:
  0: chair
  1: table
  2: sofa
  3: bed
  4: tv
  5: bin
  6: fridge
1998


In [ ]:
# 3. Fine-tune YOLO26n from COCO weights, with an overall progress line after every epoch
import time
from ultralytics import YOLO

model = YOLO('yolo26n.pt')
t0 = time.time()

def show_progress(trainer):
    """Print overall progress, ETA and val metrics after each epoch."""
    ep, total = trainer.epoch + 1, trainer.epochs          # trainer.epoch counts from 0
    elapsed = time.time() - t0
    eta = elapsed / ep * (total - ep)
    m = trainer.metrics                                    # val results of this epoch
    print(f'>>> epoch {ep}/{total} ({100 * ep / total:.0f}%) | '
          f'mAP50 {m.get("metrics/mAP50(B)", 0):.3f} | '
          f'mAP50-95 {m.get("metrics/mAP50-95(B)", 0):.3f} | '
          f'elapsed {elapsed / 60:.1f} min | ETA {eta / 60:.1f} min')

model.add_callback('on_fit_epoch_end', show_progress)
model.train(data='/content/house_yolo/data.yaml', epochs=100, imgsz=320, batch=32,
            patience=20, project='/content/runs', name='house_v1')


Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings
Ultralytics 8.4.215 🚀 Python-3.13.16 torch-2.14.1+cu130 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/house_yolo/data.yaml, degrees=0.0, deterministic=True, device=, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, 

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0, 1, 2, 3, 4, 5, 6])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x78c9ac3b2d60>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046,    0.047047,
  

In [ ]:
# 4. Final check on the test split (unseen 'dim' lighting + unseen positions)
best = YOLO('/content/runs/house_v1/weights/best.pt')
best.val(data='/content/house_yolo/data.yaml', split='test', project='/content/runs', name='house_v1_test')


Ultralytics 8.4.215 🚀 Python-3.13.16 torch-2.14.1+cu130 CUDA:0 (Tesla T4, 14913MiB)
YOLO26n summary (fused): 121 layers, 2,391,193 parameters, 0 gradients, 5.4 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 2715.8±1004.3 MB/s, size: 148.9 KB)
val: Scanning /content/house_yolo/labels/test... 507 images, 51 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 507/507 1.2Kit/s 0.4s
val: New cache created: /content/house_yolo/labels/test.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 32/32 5.3it/s 6.1s
                   all        507       1396       0.96      0.932       0.97      0.873
                 chair        267        578      0.955      0.905      0.947      0.816
                 table        217        263      0.915      0.935      0.955      0.836
                  sofa        172        172      0.964      0.944       0.97      0.911
                   bed         55         55       0.94      0.858      0.

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0, 1, 2, 3, 4, 5, 6])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x78cafb2c6740>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046,    0.047047,
  

In [ ]:
# 5. Pack the whole run (weights, plots, results.csv, test plots) and download it
!cd /content/runs && zip -rq /content/house_v1.zip house_v1 house_v1_test
from google.colab import files
files.download('/content/house_v1.zip')


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>